In [4]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
import numpy as np
import requests
from scipy.stats import pearsonr, spearmanr
from io import StringIO, BytesIO
import zipfile


# ── Paramètres globaux ─────────────────────────────────────────────────────────
# dpi=120 : résolution des figures (120 points par pouce, plus net que le défaut)
plt.rcParams["figure.dpi"] = 120
# whitegrid : fond blanc avec grille légère ; palette "muted" : couleurs douces
sns.set_theme(style="whitegrid", palette="muted")

# URL du fond de carte GeoJSON 
# contours des 96 départements métropolitains
# 
GEOJSON_URL = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"

Import de la base de données avec les déplacements domicile - travail sur le site de l'Insee

In [5]:

url = "https://www.data.gouv.fr/api/1/datasets/r/b35881a9-da09-49bf-a80e-8fa17651e927"

try:
    # 1. On télécharge le ZIP
    response = requests.get(url)
    
    # 2. On ouvre l'archive en mémoire
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        # 3. On ouvre spécifiquement le fichier de données
        with z.open('DS_RP_NAVETTES_PRINC_2022_data.csv') as f:
            df_transport = pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False)
            
    print(f"✅ Importation réussie ! {len(df_transport)} lignes chargées.")
    print(df_transport.head())

except Exception as e:
    print(f"❌ Erreur : {e}")

✅ Importation réussie ! 1191432 lignes chargées.
      AGE  EMPSTA_ENQ FREQ        GEO GEO_OBJECT RP_MEASURE TRANS WORK_AREA  \
0  Y_GE15           1    A          F     FRANCE        POP     2        _T   
1  Y_GE15           1    A  249710047       EPCI        POP    _T        10   
2  Y_GE15           1    A  249710047       EPCI        POP     6        _T   
3  Y_GE15           1    A          F     FRANCE        POP     1        _T   
4  Y_GE15           1    A         FM     FRANCE        POP    _T        23   

   TIME_PERIOD     OBS_VALUE  
0         2022  1.729514e+06  
1         2022  2.045000e+03  
2         2022  1.310000e+02  
3         2016  1.153684e+06  
4         2011  9.959466e+05  


Vérifications et prise en main de la base 

In [30]:
print(df_transport.info())
print(df_transport.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1191432 entries, 0 to 1191431
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   AGE          1191432 non-null  object 
 1   EMPSTA_ENQ   1191432 non-null  int64  
 2   FREQ         1191432 non-null  object 
 3   GEO          1191432 non-null  object 
 4   GEO_OBJECT   1191432 non-null  object 
 5   RP_MEASURE   1191432 non-null  object 
 6   TRANS        1191432 non-null  object 
 7   WORK_AREA    1191432 non-null  object 
 8   TIME_PERIOD  1191432 non-null  int64  
 9   OBS_VALUE    1191432 non-null  float64
dtypes: float64(1), int64(2), object(7)
memory usage: 90.9+ MB
None
AGE                 1
EMPSTA_ENQ          1
FREQ                1
GEO             37894
GEO_OBJECT         11
RP_MEASURE          1
TRANS               8
WORK_AREA           7
TIME_PERIOD         3
OBS_VALUE      804699
dtype: int64


In [7]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [9]:
# URL du fichier Excel sur Onnyxia
url = "https://minio.lab.sspcloud.fr/teodora/epci_2022.xlsx"

try:
    # Lecture du fichier
    epci22 = pd.read_excel(url)
    
    print("✅ Importation réussie !")
    # Affichage des premières lignes pour vérifier
    print(epci22.head())
    
except Exception as e:
    print(f"❌ Erreur : {e}")

✅ Importation réussie !
  CODGEO                   LIBGEO       EPCI                   LIBEPCI DEP  \
0  01001  L'Abergement-Clémenciat  200069193           CC de la Dombes  01   
1  01002    L'Abergement-de-Varey  240100883  CC de la Plaine de l'Ain  01   
2  01004        Ambérieu-en-Bugey  240100883  CC de la Plaine de l'Ain  01   
3  01005      Ambérieux-en-Dombes  200042497    CC Dombes Saône Vallée  01   
4  01006                  Ambléon  200040350              CC Bugey Sud  01   

   REG  
0   84  
1   84  
2   84  
3   84  
4   84  


In [32]:
# Liste des colonnes à analyser
colonnes_cibles = ['WORK_AREA', 'TIME_PERIOD', 'TRANS']

for col in colonnes_cibles:
    print(f"--- Modalités pour {col} ---")
    # Affiche les valeurs uniques
    print(df_transport[col].unique())
    # Affiche le nombre d'occurrences pour chaque modalité
    # print(df[col].value_counts())
    print("\n")

--- Modalités pour WORK_AREA ---
['_T' '10' '23' '22' '21' '20_30' '24T30']


--- Modalités pour TIME_PERIOD ---
[2022 2016 2011]


--- Modalités pour TRANS ---
['2' '_T' '6' '1' '3' '4' '5' '3T4']




analyse des modes de transport 

In [45]:
print(df_transport.columns)
print("Modalités de TRANS :", df_transport['TRANS'].unique())
print("Modalités de WORK_AREA :", df_transport['WORK_AREA'].unique())
print("Modalités croisées :", pd.crosstab(df_transport['TRANS'], df_transport['WORK_AREA']))

Index(['AGE', 'EMPSTA_ENQ', 'FREQ', 'GEO', 'GEO_OBJECT', 'RP_MEASURE', 'TRANS',
       'WORK_AREA', 'TIME_PERIOD', 'OBS_VALUE'],
      dtype='object')
Modalités de TRANS : ['2' '_T' '6' '1' '3' '4' '5' '3T4']
Modalités de WORK_AREA : ['_T' '10' '23' '22' '21' '20_30' '24T30']
Modalités croisées : WORK_AREA      10   20_30      21      22      23  24T30      _T
TRANS                                                           
1               0       0       0       0       0      0   81651
2               0       0       0       0       0      0   76071
3               0       0       0       0       0      0   28911
3T4             0       0       0       0       0      0   34571
4               0       0       0       0       0      0   31321
5               0       0       0       0       0      0   83507
6               0       0       0       0       0      0   68824
_T         124997  125248  125133  111397  111954  62576  125271


In [42]:
# 1. Préparation
df_clean = df_transport.copy()
df_clean = df_clean.rename(columns={'OBS_VALUE': 'VALUE'})
df_clean['VALUE'] = pd.to_numeric(df_clean['VALUE'], errors='coerce')

print(df_clean.head(5))



      AGE  EMPSTA_ENQ FREQ        GEO GEO_OBJECT RP_MEASURE TRANS WORK_AREA  \
0  Y_GE15           1    A          F     FRANCE        POP     2        _T   
1  Y_GE15           1    A  249710047       EPCI        POP    _T        10   
2  Y_GE15           1    A  249710047       EPCI        POP     6        _T   
3  Y_GE15           1    A          F     FRANCE        POP     1        _T   
4  Y_GE15           1    A         FM     FRANCE        POP    _T        23   

   TIME_PERIOD         VALUE  
0         2022  1.729514e+06  
1         2022  2.045000e+03  
2         2022  1.310000e+02  
3         2016  1.153684e+06  
4         2011  9.959466e+05  


In [48]:
# modes de transport par période 

# 1. TIME_PERIOD vs WORK_AREA
tab_work_area = pd.crosstab(
    df_transport['TIME_PERIOD'], 
    df_transport['WORK_AREA'], 
    dropna=False
)

# 2. TIME_PERIOD vs TRANS
tab_trans = pd.crosstab(
    df_transport['TIME_PERIOD'], 
    df_transport['TRANS'], 
    dropna=False
)

print("--- Tableau Croisé : Année vs Zone de Travail ---")
print(tab_work_area)

print("\n--- Tableau Croisé : Année vs Mode de Transport ---")
print(tab_trans)

--- Tableau Croisé : Année vs Zone de Travail ---
WORK_AREA       10  20_30     21     22     23  24T30      _T
TIME_PERIOD                                                  
2011         41675  41750  41712  35404  38099  20376   41759
2016         41674  41750  41710  37840  36706  21010  231585
2022         41648  41748  41711  38153  37149  21190  256783

--- Tableau Croisé : Année vs Mode de Transport ---
TRANS            1      2      3    3T4      4      5      6      _T
TIME_PERIOD                                                         
2011             0      0      0      0      0      0      0  260775
2016         40880  38200      0  34571      0  41753  34425  262446
2022         40771  37871  28911      0  31321  41754  34399  263355


Onn va utiliser que 2016 et 2022, pour 2011 nous n'avons pas le détail des modes de transport 

In [46]:


# --- ANALYSE A : ÉVOLUTION DES MODES DE TRANSPORT (Global France) ---
# On filtre : WORK_AREA est total, TRANS ne l'est pas, et on cible la France (GEO == 'F' ou 'FM')
df_trans = df_clean[(df_clean['WORK_AREA'] == '_T') & 
                    (df_clean['TRANS'] != '_T') & 
                    (df_clean['GEO'].isin(['F', 'FM']))]

evol_trans = df_trans.pivot_table(index='TIME_PERIOD', columns='TRANS', values='VALUE', aggfunc='sum')
# Conversion en % par année pour voir l'évolution des comportements
evol_trans_pct = evol_trans.div(evol_trans.sum(axis=1), axis=0) * 100

print("--- ÉVOLUTION DE LA PART DES MODES DE TRANSPORT (%) ---")
print(evol_trans_pct.round(1))

# --- ANALYSE B : ÉVOLUTION DES DESTINATIONS ---
# On filtre : TRANS est total, WORK_AREA ne l'est pas
df_area = df_clean[(df_clean['TRANS'] == '_T') & 
                   (df_clean['WORK_AREA'] != '_T') & 
                   (df_clean['GEO'].isin(['F', 'FM']))]

evol_area = df_area.pivot_table(index='TIME_PERIOD', columns='WORK_AREA', values='VALUE', aggfunc='sum')
evol_area_pct = evol_area.div(evol_area.sum(axis=1), axis=0) * 100

print("\n--- ÉVOLUTION DE LA DISTANCE DOMICILE-TRAVAIL (%) ---")
print(evol_area_pct.round(1))

--- ÉVOLUTION DE LA PART DES MODES DE TRANSPORT (%) ---
TRANS          1    2    3  3T4    4     5     6
TIME_PERIOD                                     
2016         4.3  6.3  NaN  3.9  NaN  70.3  15.1
2022         4.1  6.1  3.3  NaN  1.8  69.4  15.3

--- ÉVOLUTION DE LA DISTANCE DOMICILE-TRAVAIL (%) ---
WORK_AREA      10  20_30    21   22   23  24T30
TIME_PERIOD                                    
2011         20.8   39.6  28.6  7.8  2.3    0.9
2016         20.7   39.7  28.5  8.3  1.9    1.0
2022         19.4   40.3  28.7  8.4  2.1    1.1


In [47]:
# 2. On cible les données Nationales (élargi pour 2011)
# On accepte 'F', 'FM' et on vérifie si d'autres codes globaux existent
df_nat = df_clean[(df_clean['WORK_AREA'] == '_T') & (df_clean['TRANS'] != '_T')]

# 3. Pivot table brut
evol = df_nat.pivot_table(index='TIME_PERIOD', columns='TRANS', values='VALUE', aggfunc='sum')

# 4. RECONSTRUCTION DE LA MODALITÉ 3T4 (Voiture)
# Si 3 et 4 existent séparément, on les additionne dans 3T4 et on les supprime
if 3 in evol.columns and 4 in evol.columns:
    evol['3T4'] = evol['3T4'].fillna(0) + evol[3].fillna(0) + evol[4].fillna(0)
    evol = evol.drop(columns=[3, 4])

# 5. Calcul des pourcentages
evol_pct = evol.div(evol.sum(axis=1), axis=0) * 100

print("--- ÉVOLUTION CORRIGÉE DES MODES DE TRANSPORT (%) ---")
print(evol_pct.round(1))

--- ÉVOLUTION CORRIGÉE DES MODES DE TRANSPORT (%) ---
TRANS          1    2    3  3T4    4     5     6
TIME_PERIOD                                     
2016         4.2  6.4  NaN  4.0  NaN  69.8  15.6
2022         4.1  6.2  3.4  NaN  1.8  68.8  15.8
